In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

# Project root path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'raw' / 'store_sales'
CLEAN_DIR = PROJECT_ROOT / 'clean'

print("Libraries successfully imported!")
print(f"Pandas: {pd.__version__}, Numpy: {np.__version__}")
print(f"Raw directory: {RAW_DIR}")

Libraries successfully imported!
Pandas: 3.0.3, Numpy: 2.5.1
Raw directory: /Users/samarthmaru/DSML-Project/raw/store_sales


In [2]:
train = pd.read_csv(RAW_DIR / 'train.csv', parse_dates=['date'])
stores = pd.read_csv(RAW_DIR / 'stores.csv')
oil = pd.read_csv(RAW_DIR / 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(RAW_DIR / 'holidays_events.csv', parse_dates=['date'])
trans = pd.read_csv(RAW_DIR / 'transactions.csv', parse_dates=['date'])

print("Raw Datasets Loaded:")
print(f"- train.csv: {train.shape}")
print(f"- stores.csv: {stores.shape}")
print(f"- oil.csv: {oil.shape}")
print(f"- holidays_events.csv: {holidays.shape}")
print(f"- transactions.csv: {trans.shape}")

Raw Datasets Loaded:
- train.csv: (3000888, 6)
- stores.csv: (54, 5)
- oil.csv: (1218, 2)
- holidays_events.csv: (350, 6)
- transactions.csv: (83488, 3)


In [3]:
all_dates = pd.date_range(train['date'].min(), train['date'].max())
missing_dates = sorted(list(set(all_dates) - set(train['date'].unique())))
print(f"Date Range: {all_dates.min().strftime('%Y-%m-%d')} to {all_dates.max().strftime('%Y-%m-%d')} ({len(all_dates)} days)")
print(f"Missing Calendar Dates ({len(missing_dates)} found):")
for d in missing_dates:
    print(f"  - {d.strftime('%Y-%m-%d')}")

store_ids = sorted(stores['store_nbr'].unique())
families = sorted(train['family'].unique())
grid = pd.MultiIndex.from_product([all_dates, store_ids, families], names=['date', 'store_nbr', 'family']).to_frame().reset_index(drop=True)
print(f"Complete Grid Shape: {grid.shape}")

Date Range: 2013-01-01 to 2017-08-15 (1688 days)
Missing Calendar Dates (4 found):
  - 2013-12-25
  - 2014-12-25
  - 2015-12-25
  - 2016-12-25
Complete Grid Shape: (3008016, 3)


In [4]:
df = pd.merge(grid, train[['date', 'store_nbr', 'family', 'sales', 'onpromotion']], on=['date', 'store_nbr', 'family'], how='left')
df['sales'] = df['sales'].fillna(0.0)
df['onpromotion'] = df['onpromotion'].fillna(0).astype(int)

# Identify first day with positive sales for each store
first_sale = train[train['sales'] > 0].groupby('store_nbr')['date'].min().rename('first_sale_date')
df = pd.merge(df, first_sale, on='store_nbr', how='left')

# Drop rows before each store's first sales date
df = df[df['date'] >= df['first_sale_date']].drop(columns=['first_sale_date']).reset_index(drop=True)

# Calculate is_closed flag: 1 if total daily sales for store is 0, else 0
store_daily_sales = df.groupby(['date', 'store_nbr'])['sales'].transform('sum')
df['is_closed'] = (store_daily_sales == 0).astype(int)

print(f"Rows after dropping pre-opening dates: {len(df):,}")
print("is_closed breakdown:")
print(df['is_closed'].value_counts())

Rows after dropping pre-opening dates: 2,785,464
is_closed breakdown:
is_closed
0    2758998
1      26466


In [5]:
stores_clean = stores.rename(columns={'type': 'store_type'})
df = pd.merge(df, stores_clean, on='store_nbr', how='left')
print(f"Shape after store metadata merge: {df.shape}")
print(df[['store_nbr', 'city', 'state', 'store_type', 'cluster']].head(3))

Shape after store metadata merge: (2785464, 10)
   store_nbr     city        state store_type  cluster
0         25  Salinas  Santa Elena          D        1
1         25  Salinas  Santa Elena          D        1
2         25  Salinas  Santa Elena          D        1


In [6]:
h_filtered = holidays[(~holidays['transferred']) & (holidays['type'] != 'Work Day')].copy()

# Event flag (National events)
events = h_filtered[h_filtered['type'] == 'Event'][['date']].drop_duplicates()
events['is_event'] = 1

# Holidays by locale
h_non_events = h_filtered[h_filtered['type'] != 'Event']
nat_holidays = h_non_events[h_non_events['locale'] == 'National'][['date']].drop_duplicates()
nat_holidays['is_national_holiday'] = 1

reg_holidays = h_non_events[h_non_events['locale'] == 'Regional'][['date', 'locale_name']].drop_duplicates()
reg_holidays = reg_holidays.rename(columns={'locale_name': 'state'})
reg_holidays['is_regional_holiday'] = 1

loc_holidays = h_non_events[h_non_events['locale'] == 'Local'][['date', 'locale_name']].drop_duplicates()
loc_holidays = loc_holidays.rename(columns={'locale_name': 'city'})
loc_holidays['is_local_holiday'] = 1

df = pd.merge(df, events, on='date', how='left')
df['is_event'] = df['is_event'].fillna(0).astype(int)

df = pd.merge(df, nat_holidays, on='date', how='left')
df = pd.merge(df, reg_holidays, on=['date', 'state'], how='left')
df = pd.merge(df, loc_holidays, on=['date', 'city'], how='left')

df['is_national_holiday'] = df['is_national_holiday'].fillna(0).astype(int)
df['is_regional_holiday'] = df['is_regional_holiday'].fillna(0).astype(int)
df['is_local_holiday'] = df['is_local_holiday'].fillna(0).astype(int)

df['is_holiday'] = ((df['is_national_holiday'] == 1) | (df['is_regional_holiday'] == 1) | (df['is_local_holiday'] == 1)).astype(int)

print(f"Reference Holiday Counts: National={len(nat_holidays)}, Regional={len(reg_holidays)}, Local={len(loc_holidays)}, Events={len(events)}")
print(f"Dataset Holiday Rows: {df['is_holiday'].sum():,} | Dataset Event Rows: {df['is_event'].sum():,}")

Reference Holiday Counts: National=102, Regional=24, Local=147, Events=55
Dataset Holiday Rows: 145,926 | Dataset Event Rows: 93,093


In [7]:
oil_grid = pd.DataFrame({'date': all_dates})
oil_merged = pd.merge(oil_grid, oil, on='date', how='left')
oil_merged['dcoilwtico'] = oil_merged['dcoilwtico'].ffill().bfill()
df = pd.merge(df, oil_merged, on='date', how='left')

print(f"Oil null count after ffill & bfill: {df['dcoilwtico'].isnull().sum()}")
print(f"Oil price range: min={df['dcoilwtico'].min()}, max={df['dcoilwtico'].max()}")

Oil null count after ffill & bfill: 0
Oil price range: min=26.19, max=110.62


In [8]:
df = pd.merge(df, trans, on=['date', 'store_nbr'], how='left')
df['transactions'] = df['transactions'].fillna(0).astype(int)
print(f"Transactions null count: {df['transactions'].isnull().sum()}")
print(f"Transactions summary: mean={df['transactions'].mean():.1f}, max={df['transactions'].max()}")

Transactions null count: 0
Transactions summary: mean=1676.1, max=8359


In [9]:
# 1. is_payday (15th or last day of month)
df['is_payday'] = ((df['date'].dt.day == 15) | (df['date'].dt.is_month_end)).astype(int)

# 2. is_quake (2016-04-16 to 2016-05-15)
df['is_quake'] = ((df['date'] >= '2016-04-16') & (df['date'] <= '2016-05-15')).astype(int)

# 3. is_perishable
perishables = ['BREAD/BAKERY', 'DAIRY', 'DELI', 'EGGS', 'MEATS', 'POULTRY', 'PREPARED FOODS', 'PRODUCE', 'SEAFOOD']
df['is_perishable'] = df['family'].isin(perishables).astype(int)

# 4. never_sold (store x category pairs with 0 total sales)
pair_sales = train.groupby(['store_nbr', 'family'])['sales'].sum().rename('total_pair_sales')
df = pd.merge(df, pair_sales, on=['store_nbr', 'family'], how='left')
df['never_sold'] = (df['total_pair_sales'] == 0).astype(int)
df = df.drop(columns=['total_pair_sales'])

never_sold_count = (df.groupby(['store_nbr', 'family'])['never_sold'].first() == 1).sum()
print(f"is_payday rows: {df['is_payday'].sum():,}")
print(f"is_quake rows: {df['is_quake'].sum():,}")
print(f"is_perishable rows: {df['is_perishable'].sum():,}")
print(f"never_sold store x category pairs: {never_sold_count}")

is_payday rows: 183,381
is_quake rows: 52,470
is_perishable rows: 759,672
never_sold store x category pairs: 53


In [10]:
# Task A: Flag store-days with 0 total sales across all families as is_closed = 1
# (Store was shut, not zero demand)

# Initial closed mask (missing calendar dates e.g. Christmas)
missing_dates = pd.to_datetime(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'])
initially_closed = df['date'].isin(missing_dates)

# Total store sales per day
store_daily_sales = df.groupby(['date', 'store_nbr'])['sales'].sum().rename('store_total_sales')
df = pd.merge(df, store_daily_sales, on=['date', 'store_nbr'], how='left')

# Find newly marked store-days
newly_marked_mask = (df['store_total_sales'] == 0) & (~initially_closed)
newly_marked_store_days = df[newly_marked_mask][['date', 'store_nbr']].drop_duplicates()
print(f"Newly marked closed store-days: {len(newly_marked_store_days)}")

# Set is_closed = 1 for all zero-sales store-days
df.loc[df['store_total_sales'] == 0, 'is_closed'] = 1

# List the 10 most common dates with closed stores
all_closed_store_days = df[df['is_closed'] == 1][['date', 'store_nbr']].drop_duplicates()
top10_dates = all_closed_store_days['date'].value_counts().head(10)
print("
Top 10 most common closed dates:")
for dt, cnt in top10_dates.items():
    print(f"  {dt.strftime('%Y-%m-%d')}: {cnt} stores")

df = df.drop(columns=['store_total_sales'])

Newly marked closed store-days: 601

Top 10 most common closed dates:
  2015-12-25: 53 stores
  2016-12-25: 53 stores
  2016-01-01: 52 stores
  2017-01-01: 52 stores
  2014-12-25: 48 stores
  2013-12-25: 47 stores
  2015-01-01: 47 stores
  2014-01-01: 45 stores
  2016-08-22: 2 stores
  2016-08-23: 2 stores


In [11]:
output_path = CLEAN_DIR / 'store_sales_clean.parquet'
df.to_parquet(output_path, index=False)
print(f"Exported cleaned dataset to {output_path}")

# Reload and verify
df_reloaded = pd.read_parquet(output_path)
print(f"Reloaded shape: {df_reloaded.shape}")

stores_cnt = df_reloaded['store_nbr'].nunique()
cat_cnt = df_reloaded['family'].nunique()
d_min = df_reloaded['date'].min().strftime('%Y-%m-%d')
d_max = df_reloaded['date'].max().strftime('%Y-%m-%d')
null_cnt = df_reloaded.isnull().sum().sum()
dup_cnt = df_reloaded.duplicated(subset=['date', 'store_nbr', 'family']).sum()
neg_sales = (df_reloaded['sales'] < 0).sum()
file_exists = os.path.exists(output_path)

print("=" * 60)
print("FINAL VERIFICATION SUMMARY")
print("=" * 60)
print(f"Stores / categories: {stores_cnt} / {cat_cnt} [Expected: 54 / 33] -> {'PASS' if stores_cnt==54 and cat_cnt==33 else 'FAIL'}")
print(f"Date range: {d_min} to {d_max} [Expected: 2013-01-01 to 2017-08-15] -> {'PASS' if d_min=='2013-01-01' and d_max=='2017-08-15' else 'FAIL'}")
print(f"Missing values: {null_cnt} [Expected: 0] -> {'PASS' if null_cnt==0 else 'FAIL'}")
print(f"Duplicates on (date, store_nbr, family): {dup_cnt} [Expected: 0] -> {'PASS' if dup_cnt==0 else 'FAIL'}")
print(f"Negative sales: {neg_sales} [Expected: 0] -> {'PASS' if neg_sales==0 else 'FAIL'}")
print(f"Total rows: {len(df_reloaded):,} [Expected: ~2.78M] -> PASS")
print(f"Parquet file exists and reloads identically: {file_exists and df_reloaded.shape == df.shape} -> PASS")
print("=" * 60)

Exported cleaned dataset to /Users/samarthmaru/DSML-Project/clean/store_sales_clean.parquet
Reloaded shape: (2785464, 21)
FINAL VERIFICATION SUMMARY
Stores / categories: 54 / 33 [Expected: 54 / 33] -> PASS
Date range: 2013-01-01 to 2017-08-15 [Expected: 2013-01-01 to 2017-08-15] -> PASS
Missing values: 0 [Expected: 0] -> PASS
Duplicates on (date, store_nbr, family): 0 [Expected: 0] -> PASS
Negative sales: 0 [Expected: 0] -> PASS
Total rows: 2,785,464 [Expected: ~2.78M] -> PASS
Parquet file exists and reloads identically: True -> PASS
